# 01. Measles TCR-beta analysis

Clinically corrected patient- and group-level analysis of productive human TRB repertoires. Clinical T0/T1/T2/T3 values come from the curated metadata and are not inferred from the raw .1/.2/.3 suffix.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd().resolve()
os.environ['MEASLES_REPO'] = str(ROOT)
print('Notebook root:', ROOT)

def run_script(name):
    path = ROOT / 'scripts' / name
    print('RUN', path)
    exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), globals())


## 1. Build biological samples and normalize repertoire depth


In [ ]:
run_script('analysis_01_prepare.py')
for p in [ROOT/'results'/'combined_trb_clonotypes.csv', ROOT/'results'/'diversity_metrics.csv']:
    if not p.exists() or p.stat().st_size == 0:
        raise RuntimeError(f'Missing analysis prerequisite: {p}')


## 2. Exact specificity reference audit

Exact IEDB matching is retained as a reference-coverage/QC step. The corrected summary does not interpret a zero exact-match result as biological absence when the reference itself has no sequences for a target pathogen.

In [ ]:
run_script('analysis_02_antigens.py')


## 3. Raw FASTQ depth QC

Count total input read pairs once. This does not rerun MiXCR. Existing `results/raw_fastq_depth.csv` is reused on later runs.

In [ ]:
raw_depth = ROOT / 'results' / 'raw_fastq_depth.csv'
if raw_depth.exists() and raw_depth.stat().st_size > 0:
    print('Reuse', raw_depth)
else:
    subprocess.run([sys.executable, str(ROOT/'scripts'/'collect_fastq_depth.py'), '--workers', '8'], check=True)


## 4. Corrected clinical summary, QC, statistics and figures

This is the authoritative downstream analysis. It applies the curated clinical phases, excludes failed/suspect libraries from biological inference, keeps adults and children separate, separates old/new adult batches, and reports effect sizes with FDR.

In [ ]:
run_script('analysis_06_clinical_summary.py')


In [ ]:
required = [
 ROOT/'results'/'corrected'/'sample_metadata_enriched_with_analysis.csv',
 ROOT/'results'/'corrected'/'cross_sectional_statistics.csv',
 ROOT/'results'/'corrected'/'longitudinal_statistics.csv',
 ROOT/'results'/'corrected'/'T1_T2_repertoire_similarity.csv',
 ROOT/'results'/'corrected'/'TRBV_T1_T2_paired_statistics.csv',
 ROOT/'results'/'corrected'/'specificity_reference_coverage.csv',
 ROOT/'figures'/'corrected'/'01_oligoclonality_case_control.png',
 ROOT/'figures'/'corrected'/'05_richness_T1_T2.png',
 ROOT/'figures'/'corrected'/'08_weighted_similarity_T1_T2.png',
]
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Corrected analysis finished without required outputs: {missing}')
print('Corrected analysis output check passed.')
